# RoboCar MCP Server — Google Colab

Notebook ini menjalankan MCP Server RoboCar di Google Colab. Karena Colab
tidak mendukung transport **stdio** (butuh proses yang di-spawn lokal oleh
host), server di sini pakai transport **streamable-http** dan di-expose ke
internet lewat **ngrok**, digabung dalam satu port yang sama dengan REST
API + dashboard visual.

Urutan menjalankan: jalankan tiap cell dari atas ke bawah secara berurutan.

In [ ]:
!pip install -q mcp fastapi uvicorn httpx pyngrok nest_asyncio

## 1. Buat struktur folder proyek

In [ ]:
import os
os.makedirs("mcp_server", exist_ok=True)
open("mcp_server/__init__.py", "w").close()
print("Folder siap.")

## 2. Tulis file `state.py`

In [ ]:
%%writefile mcp_server/state.py
import math
import threading
import time
from typing import Any, Dict, List


class RobocarState:
    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._state: Dict[str, Any] = {
            "x": 0.0,
            "y": 0.0,
            "heading": 0.0,
            "speed": 0.0,
            "battery": 100.0,
        }
        self._commands: List[Dict[str, Any]] = []
        self._next_id = 1

    def get_state(self) -> Dict[str, Any]:
        with self._lock:
            return dict(self._state)

    def apply_command(self, tool_name: str, params: Dict[str, Any]) -> Dict[str, Any]:
        with self._lock:
            if tool_name == "move_forward":
                distance = float(params.get("distance", 0))
                rad = math.radians(self._state["heading"])
                self._state["x"] += distance * math.cos(rad)
                self._state["y"] += distance * math.sin(rad)
                self._state["speed"] = distance
                self._state["battery"] = max(0.0, self._state["battery"] - distance * 0.5)
            elif tool_name == "move_backward":
                distance = float(params.get("distance", 0))
                rad = math.radians(self._state["heading"])
                self._state["x"] -= distance * math.cos(rad)
                self._state["y"] -= distance * math.sin(rad)
                self._state["speed"] = -distance
                self._state["battery"] = max(0.0, self._state["battery"] - distance * 0.5)
            elif tool_name == "turn_left":
                degree = float(params.get("degree", 0))
                self._state["heading"] = (self._state["heading"] + degree) % 360
            elif tool_name == "turn_right":
                degree = float(params.get("degree", 0))
                self._state["heading"] = (self._state["heading"] - degree) % 360
            elif tool_name == "stop":
                self._state["speed"] = 0.0
            elif tool_name == "get_status":
                pass
            else:
                raise ValueError(f"Unknown tool: {tool_name}")

            entry = {
                "id": self._next_id,
                "timestamp": time.time(),
                "tool": tool_name,
                "params": params,
                "result_state": dict(self._state),
            }
            self._commands.append(entry)
            self._next_id += 1
            return entry

    def get_commands_since(self, since_id: int = 0) -> List[Dict[str, Any]]:
        with self._lock:
            return [c for c in self._commands if c["id"] > since_id]


robocar_state = RobocarState()


## 3. Tulis file `tools.py`

In [ ]:
%%writefile mcp_server/tools.py
from mcp.server import MCPServer

from .state import robocar_state

mcp = MCPServer("robocar-mcp")


@mcp.tool()
def move_forward(distance: float) -> dict:
    """Gerakkan robocar maju sejauh `distance` unit."""
    entry = robocar_state.apply_command("move_forward", {"distance": distance})
    return entry["result_state"]


@mcp.tool()
def move_backward(distance: float) -> dict:
    """Gerakkan robocar mundur sejauh `distance` unit."""
    entry = robocar_state.apply_command("move_backward", {"distance": distance})
    return entry["result_state"]


@mcp.tool()
def turn_left(degree: float) -> dict:
    """Putar robocar ke kiri (berlawanan arah jarum jam) sejauh `degree` derajat."""
    entry = robocar_state.apply_command("turn_left", {"degree": degree})
    return entry["result_state"]


@mcp.tool()
def turn_right(degree: float) -> dict:
    """Putar robocar ke kanan (searah jarum jam) sejauh `degree` derajat."""
    entry = robocar_state.apply_command("turn_right", {"degree": degree})
    return entry["result_state"]


@mcp.tool()
def stop() -> dict:
    """Hentikan robocar (speed = 0)."""
    entry = robocar_state.apply_command("stop", {})
    return entry["result_state"]


@mcp.tool()
def get_status() -> dict:
    """Ambil state terkini robocar (posisi, arah, speed, baterai)."""
    entry = robocar_state.apply_command("get_status", {})
    return entry["result_state"]


## 4. Tulis file `dashboard.py`

In [ ]:
%%writefile mcp_server/dashboard.py
DASHBOARD_HTML = """<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<title>RoboCar Simulator Dashboard</title>
<style>
  body { background:#111; color:#eee; font-family: monospace; text-align:center; }
  canvas { background:#1c1c1c; border:1px solid #444; margin-top:20px; }
  #info { margin-top:12px; font-size:14px; }
  #info span { color:#4ade80; }
  h2 { color:#eee; }
</style>
</head>
<body>
  <h2>RoboCar Simulator Dashboard</h2>
  <canvas id="canvas" width="600" height="600"></canvas>
  <div id="info">Menyambung...</div>

<script>
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');
const info = document.getElementById('info');
const SCALE = 15;
const CENTER = 300;
let trail = [];

function drawGrid() {
  ctx.strokeStyle = '#2a2a2a';
  ctx.lineWidth = 1;
  for (let i = 0; i <= 600; i += SCALE * 2) {
    ctx.beginPath(); ctx.moveTo(i, 0); ctx.lineTo(i, 600); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(0, i); ctx.lineTo(600, i); ctx.stroke();
  }
  ctx.strokeStyle = '#555';
  ctx.beginPath(); ctx.moveTo(CENTER, 0); ctx.lineTo(CENTER, 600); ctx.stroke();
  ctx.beginPath(); ctx.moveTo(0, CENTER); ctx.lineTo(600, CENTER); ctx.stroke();
}

function drawCar(x, y, heading) {
  const px = CENTER + x * SCALE;
  const py = CENTER - y * SCALE;
  const rad = -heading * Math.PI / 180;

  trail.push([px, py]);
  if (trail.length > 200) trail.shift();
  ctx.strokeStyle = '#3b82f6';
  ctx.beginPath();
  trail.forEach(([tx, ty], i) => i === 0 ? ctx.moveTo(tx, ty) : ctx.lineTo(tx, ty));
  ctx.stroke();

  ctx.save();
  ctx.translate(px, py);
  ctx.rotate(rad);
  ctx.fillStyle = '#4ade80';
  ctx.beginPath();
  ctx.moveTo(14, 0);
  ctx.lineTo(-10, -8);
  ctx.lineTo(-10, 8);
  ctx.closePath();
  ctx.fill();
  ctx.restore();
}

async function poll() {
  try {
    const res = await fetch('/state');
    const s = await res.json();
    ctx.clearRect(0, 0, 600, 600);
    drawGrid();
    drawCar(s.x, s.y, s.heading);
    info.innerHTML = `x: <span>${s.x.toFixed(2)}</span> | y: <span>${s.y.toFixed(2)}</span> | heading: <span>${s.heading.toFixed(1)}&deg;</span> | speed: <span>${s.speed}</span> | battery: <span>${s.battery.toFixed(1)}%</span>`;
  } catch (e) {
    info.textContent = 'Gagal konek ke server...';
  }
}

drawGrid();
setInterval(poll, 500);
poll();
</script>
</body>
</html>
"""


## 5. Tulis file `api.py`

In [ ]:
%%writefile mcp_server/api.py
from fastapi import FastAPI, Query
from fastapi.responses import HTMLResponse

from .dashboard import DASHBOARD_HTML
from .state import robocar_state

api = FastAPI(title="RoboCar MCP API")


@api.get("/", response_class=HTMLResponse)
def dashboard():
    return DASHBOARD_HTML


@api.get("/state")
def get_state():
    return robocar_state.get_state()


@api.get("/commands")
def get_commands(since_id: int = Query(0, description="Command dengan id lebih besar dari nilai ini")):
    return robocar_state.get_commands_since(since_id)


## 6. Setup token ngrok

Daftar akun gratis di [ngrok.com](https://ngrok.com), ambil authtoken dari
dashboard, lalu isi di bawah (atau simpan sebagai Colab Secret bernama
`NGROK_TOKEN` lewat ikon kunci di sidebar kiri).

In [ ]:
from google.colab import userdata

try:
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = "ISI_AUTHTOKEN_NGROK_DI_SINI"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

## 7. Jalankan Server (API + Dashboard + MCP streamable-http)

MCP app di-mount ke path `/mcp-app` pada FastAPI app yang sama, jadi cukup
satu port dan satu tunnel ngrok buat semuanya.

In [ ]:
import contextlib
import threading
import time

import nest_asyncio
import uvicorn
from mcp.server.transport_security import TransportSecuritySettings

nest_asyncio.apply()

from mcp_server.api import api
from mcp_server.tools import mcp

# enable_dns_rebinding_protection=False: server ini sengaja di-expose publik
# lewat ngrok, jadi proteksi DNS-rebinding perlu dimatikan supaya client dari
# domain manapun (misal web-based MCP Inspector) bisa connect.
mcp_app = mcp.streamable_http_app(
    transport_security=TransportSecuritySettings(enable_dns_rebinding_protection=False)
)
api.mount("/mcp-app", mcp_app)

# Gabungkan lifespan supaya session manager MCP ikut ter-start bareng FastAPI
# (tanpa ini, endpoint /mcp-app/mcp akan error 500 "Task group is not initialized")
_original_lifespan = api.router.lifespan_context


@contextlib.asynccontextmanager
async def _combined_lifespan(app):
    async with contextlib.AsyncExitStack() as stack:
        await stack.enter_async_context(mcp_app.router.lifespan_context(mcp_app))
        async with _original_lifespan(app):
            yield


api.router.lifespan_context = _combined_lifespan


def run_server():
    uvicorn.run(api, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

public_url = ngrok.connect(8000)
print("Dashboard & REST API :", public_url)
print("MCP endpoint (streamable-http) :", f"{public_url}/mcp-app/mcp")

## 8. Buka Dashboard Visual

Klik link `public_url` dari cell di atas — bakal kebuka dashboard yang
sama persis kayak versi lokal (canvas + posisi robocar real-time).

## 9. Test Tools Langsung (tanpa lewat protokol MCP, buat verifikasi cepat)

In [ ]:
from mcp_server.tools import get_status, move_forward, turn_left

print(move_forward(5))
print(turn_left(90))
print(get_status())

## 10. Connect dari MCP Client Eksternal

Pakai MCP Inspector (jalan di komputer lokal, bukan di Colab) buat connect
ke server yang ada di Colab ini:

```bash
npx @modelcontextprotocol/inspector
```

Di Inspector, pilih transport **Streamable HTTP**, isi URL dengan nilai
`f"{public_url}/mcp-app/mcp"` dari cell nomor 7, lalu Connect. Tools yang
sama (`move_forward`, `turn_left`, dst) akan muncul dan bisa dipanggil,
persis seperti kalau MCP Server-nya jalan lokal.

**Catatan:** sesi Colab bersifat sementara — begitu runtime disconnect,
URL ngrok dan semua state robocar hilang. Untuk demo/sidang yang butuh
koneksi stabil ke Claude Desktop, tetap disarankan jalankan versi lokal
(lihat README utama repo).